# Lab 5 - Quality demo
Sends 7 synthetic Wikipedia events through the quality rules: 1 valid edit and 6 bad ones (empty title, missing user, bad timestamp, negative length, canary domain, broken JSON). The notebook checks that the valid event goes to Silver and the others get a list of the rules they broke.
It tests the shared routing functions only. Real Lakeflow expectation metrics for the same events come from deploying with `quality_demo=true`.

In [ ]:
%run ./lab5_00_config

In [ ]:
from lab5.fixtures import wiki_fixture_rows
fixture = spark.createDataFrame(wiki_fixture_rows(), "event_json STRING, kafka_topic STRING, kafka_partition INT, kafka_offset BIGINT, kafka_enqueued_at TIMESTAMP, _ingested_at TIMESTAMP")
prepared = annotate_quality(parse_wiki(fixture), WIKI_RULES)
valid, rejected = prepared.filter("_is_valid"), prepared.filter("NOT _is_valid")
assert valid.count() == 1
assert rejected.count() == 6
assert rejected.filter("size(_quality_errors) = 0").count() == 0
assert_equal(valid.select("kafka_offset").unionByName(rejected.select("kafka_offset")), fixture, ["kafka_offset"], "fixture reconciliation")
display(prepared.select("event_id", "kafka_offset", "_is_valid", "_quality_errors"))
